I would only use this if doing analysis on a small dataset or else it takes an ungodly amount of time to finish.

In [ ]:
import numpy as np 
import random as ran 
import matplotlib.pyplot as plt 
from scipy import stats
from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict

import sys
import re
import glob 
import os

import importlib

sys.path.insert(0, "/hepusers2/fuscomus/DRToM")
import functions as fns
importlib.reload(fns)
import configuration as cfg
importlib.reload(cfg)

import DimensionalReduction as dr
importlib.reload(dr)

LHAPDF 6.5.5 loading /hepusers2/fuscomus/miniconda3/envs/DRToM/share/LHAPDF/cteq6l1/cteq6l1_0000.dat
cteq6l1 PDF set, member #0, version 4; LHAPDF ID = 10042
LHAPDF 6.5.5 loading /hepusers2/fuscomus/miniconda3/envs/DRToM/share/LHAPDF/cteq6l1/cteq6l1_0000.dat
cteq6l1 PDF set, member #0, version 4; LHAPDF ID = 10042


<module 'DimensionalReduction' from '/hepusers2/fuscomus/DRToM/Analysis/DimensionalReduction.py'>

## Build the LHE data set 

The first thing we need to do is build the dimensional reduction data set we want to analyze. This is specific to the data file structure. 

Here we define the start and end generation scale we are looking at, the step size between generations, and the dimensional reduction scale we are looking at. Then, the code will go into the data and build the approprite 2D files (those that are in the 2D folder from the DR_scale up to the End scale), as well as the 3D files (those thare in the 3D folder from the Start scale up to the DR_scale). Combined, those files offer a proper generation range with whatever choice of DR. 

In [ ]:
base_path = "/hepusers2/fuscomus/DRToM/Data_Test"
energy_folder = "TeV13p0"    # Options: TeV13p0, TeV13p6
dimensionality_2D = "2D"
dimensionality_3D = "3D"
what_process = "2to4"        # Options: 2to2, 2to3, 2to4, 2to5

Start = 2.0                 # What is the start of the generation range (TeV)
End = 11.0                  # What is the end of the generation range (TeV)
Step = 0.1                  # What is the step size of the generation range (TeV)
DR_scale = 2.0             # Any DR value allowed (TeV)

# Construct LHE file lists split by DR_scale
def _slice_label(s, e):
    return f"{s:.1f}to{e:.1f}"

lhe_2D_files = []
lhe_3D_files = []

# For each slice (e.g. 9.0to9.1) gather matching .lhe files from 3D or 2D folders
for slice in np.arange(Start, End, Step):
    slice = round(slice, 10)  # Avoid floating point issues
    slice_end = round(slice + Step, 10)
    label = _slice_label(slice, slice_end)
    if slice < DR_scale:
        # slices below DR_scale belong to 3D
        pattern = os.path.join(base_path, energy_folder, dimensionality_3D, what_process, f"{label}_*.lhe")
        matches = sorted(glob.glob(pattern))
        if matches:
            lhe_3D_files.extend(matches)
    else:
        # slices >= DR_scale belong to 2D
        pattern = os.path.join(base_path, energy_folder, dimensionality_2D, what_process, f"{label}_*.lhe")
        matches = sorted(glob.glob(pattern))
        if matches:
            lhe_2D_files.extend(matches)

# Remove duplicates and sort
lhe_2D_files = sorted(set(lhe_2D_files))
lhe_3D_files = sorted(set(lhe_3D_files))
# print(lhe_2D_files)
# print(lhe_3D_files)

# Combined local bunch: 3D (below DR) then 2D (above DR)
local_lhe_files = lhe_3D_files + lhe_2D_files

print(f"Found {len(lhe_3D_files)} 3D LHE files and {len(lhe_2D_files)} 2D LHE files.")
print(f"Combined local bunch size: {len(local_lhe_files)} files.")

# For backward compatibility keep the tuple descriptors
lhe_2D_set = (base_path, energy_folder, dimensionality_2D, what_process)
lhe_3D_set = (base_path, energy_folder, dimensionality_3D, what_process)

# --- Helper: write a plain file list for cluster processing ---
# This writes one LHE path per line to the .txt file 
base_path = "/hepusers2/fuscomus/DRToM/Analysis"
file_list_path = os.path.join(base_path, "ClusterData", "FileLists", energy_folder, what_process, f"DR_{DR_scale}", f"{Start}_{End}_{Step}.txt")
os.makedirs(os.path.dirname(file_list_path), exist_ok=True)
with open(file_list_path, "w") as _f:
    for p in local_lhe_files:
        _f.write(p + "\n")

print(f"Wrote file list -> {file_list_path} ({len(local_lhe_files)} files)")

Found 0 3D LHE files and 3 2D LHE files.
Combined local bunch size: 3 files.
Wrote file list -> /hepusers2/fuscomus/DRToM/Analysis/ClusterData/FileLists/TeV13p0/2to4/DR_2.0/2.0_5.0_1.0.txt (3 files)


# Only run up to here if using large datasets. Switch to cluster usage. 

---

## Collect data from LHEs

Here we now run through the LHE set made above and collect the data. This accounts, in order that we plot and save after this cell, for:

- Tree level specic (e.g. individual lprup) cross sections and xa/xb values
- Invariant mass (in lab and CoM, although these should be the same) 
- Kinematics (momentum and angles) in the lab and CoM frame 
- ID counts specific to the generation 
- Event shape variables in the lab and CoM frame


Because we account for the CoM frame, this is also where we boost the lab LHE events to the CoM.

In [3]:
filenames = local_lhe_files
directories = sorted(set(os.path.dirname(f) for f in filenames))
print(f"Processing {len(filenames)} LHE files across {len(directories)} directories.")

# Storage for each directory and subprocess
data_dict = {}
event_counts = {}          # Store event counts per directory and lprup
total_cross_sections = {}  # Store total XS per directory

store_xa = []
store_xb = []

# Iterate over individual files and aggregate by their containing directory
for filename in filenames:
    #print(f"Processing file: {filename}")

    # Use the file's containing directory as the dictionary key so we can aggregate
    directory = os.path.dirname(filename)
    if directory not in data_dict:
        data_dict[directory] = {}
        event_counts[directory] = {}
        total_cross_sections[directory] = 0.0

    grouped_data = dr.read_lhe_grouped_by_lprup(filename)

    if not grouped_data:
        print(f"Warning: No event data found in {filename}")
        continue

    for lprup, info in grouped_data.items():
        events = info.get("events", [])
        xsec = info.get("cross_section", 0.0)

        if lprup not in data_dict[directory]:
            data_dict[directory][lprup] = {
                "sphericity_lab": [], "sphericity_CoM": [],
                "aplanarity_lab": [], "aplanarity_CoM": [],
                "sphericity_transverse_lab": [], "sphericity_transverse_CoM": [],
                "Y_values_lab": [], "Y_values_CoM": [],
                "C_values_lab": [], "C_values_CoM": [],
                "D_values_lab": [], "D_values_CoM": [],
                "Thrust_T_values_lab": [], "Thrust_T_values_CoM": [],
                "Thrust_m_values_lab": [], "Thrust_m_values_CoM": [],
                "tau_values_lab": [], "tau_values_CoM": [],
                "B_values_lab": [], "B_values_CoM": [],
                "energy_lab": [], "energy_CoM": [],
                "momentum_lab": [], "momentum_CoM": [],
                "pt_lab": [], "pt_CoM": [],
                "eta_lab": [], "eta_CoM": [],
                "phi_lab": [], "phi_CoM": [],
                "weighting": [],
                "M_lab": [], "M_CoM": [],
                "four_mom_lab": [], "four_mom_CoM": [],
            }
            event_counts[directory][lprup] = [0, 0]  # [files, events]

        dd = data_dict[directory][lprup]
        four_mom_lab = []
        four_mom_CoM = []

        for event in events:
            particles = event.get("final_state", [])

            # Always record the incoming parton momentum fractions
            xa = float(event.get("xa", 0.0))
            xb = float(event.get("xb", 0.0))

            # Check for unphysical values
            if xa > 1.0 or xa < 0.0 or xb > 1.0 or xb < 0.0:
                print(f"[WARNING] Unphysical momentum fraction detected: xa={xa}, xb={xb}, file={filename}")

            store_xa.append(xa)
            store_xb.append(xb)

            try:
                M_event, boosted = dr.boost_to_com(particles, debug=False)
            except ValueError as e:
                print(f"[ERROR] Boost to CoM failed for event in file {filename}, lprup {lprup}: {e}")
                continue  # Skip this event entirely

            # Store lab and CoM four-momenta per event
            four_mom_lab.append(particles)
            four_mom_CoM.append(boosted)

        # store four-momenta for this LPRUP
        dd["four_mom_lab"].extend(four_mom_lab)
        dd["four_mom_CoM"].extend(four_mom_CoM)

        # Initialize a master list for this directory if not already
        data_dict[directory].setdefault("all_four_mom_lab", []).extend(four_mom_lab)
        data_dict[directory].setdefault("all_four_mom_CoM", []).extend(four_mom_CoM)

        # === Collection ===
        # Map frames to the per-event four-momenta lists we've built above
        frames_data = {"lab": four_mom_lab, "CoM": four_mom_CoM}

        for frame, fm in frames_data.items():
            # Get kinematics (mode 'all')
            (three_mom_all, energy_list, momentum_list,
             pt_list, eta_list, phi_list,
             px_list, py_list, pz_list, theta_list,
             delta_eta_list, delta_theta_list, delta_phi_list) = dr.diff_momentum(fm, mode="all")

            # Get event shape variables
            (S, A, S_T, Y, C, D,
             Thrust_T, Thrust_m, tau, B) = dr.calc_EventVars(fm, three_mom_all, sort_by='pT', mode=1, verbose=False)

            # Apply tolerances
            A = dr.apply_tolerance(A, tol=1e-10)
            D = dr.apply_tolerance(D, tol=1e-10)
            S = dr.apply_tolerance(S, tol=1e-10)
            Y = dr.apply_tolerance(Y, tol=1e-10)
            C = dr.apply_tolerance(C, tol=1e-10)
            B = dr.apply_biplanarity_tolerance(B, tol=1e-5)

            # Suffix for keys: '_lab' or '_CoM'
            suffix = "_lab" if frame == "lab" else "_CoM"

            # Event shape vars into per-frame keys
            dd[f"sphericity{suffix}"].extend(S)
            dd[f"aplanarity{suffix}"].extend(A)
            dd[f"sphericity_transverse{suffix}"].extend(S_T)
            dd[f"Y_values{suffix}"].extend(Y)
            dd[f"C_values{suffix}"].extend(C)
            dd[f"D_values{suffix}"].extend(D)
            dd[f"Thrust_T_values{suffix}"].extend(Thrust_T)
            dd[f"Thrust_m_values{suffix}"].extend(Thrust_m)
            dd[f"tau_values{suffix}"].extend(tau)
            dd[f"B_values{suffix}"].extend(B)

            def flatten_list(list_of_lists):
                return [item for sublist in list_of_lists for item in sublist]

            # Kinematics into per-frame keys
            dd[f"energy{suffix}"].extend(flatten_list(energy_list))
            dd[f"momentum{suffix}"].extend(flatten_list(momentum_list))
            dd[f"pt{suffix}"].extend(flatten_list(pt_list))
            dd[f"eta{suffix}"].extend(eta_list)   # already flat per-event
            dd[f"phi{suffix}"].extend(phi_list)   # already flat per-event

        # Cross-section weighting + invariant mass
        n_events_lprup = len(events)
        per_event_weight = xsec

        for event in four_mom_lab:
            dd["weighting"].append(per_event_weight)
            dd["M_lab"].append(dr.invar_mass(event))

        for event in four_mom_CoM:
            dd["M_CoM"].append(dr.invar_mass(event))

        # Counts
        if n_events_lprup > 0:
            event_counts[directory][lprup][0] += 1
            event_counts[directory][lprup][1] += n_events_lprup
            total_cross_sections[directory] += xsec

Processing 450 LHE files across 1 directories.


KeyboardInterrupt: 

## xa and xb

In [ ]:
# Define bin edges with width 0.02
bins = np.arange(0, 1+ 0.02, 0.02)  # x_a and x_b must be in [0,1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Histogram for x_a 
axes[0].hist(store_xa, bins=bins, color="royalblue", alpha=0.7, edgecolor="black")
axes[0].set_yscale("log")
axes[0].set_title(r"Distribution of $x_a$", fontsize=16)
axes[0].grid(True, linestyle="--", alpha=0.6)

# Histogram for x_b
axes[1].hist(store_xb, bins=bins, color="orangered", alpha=0.7, edgecolor="black")
axes[1].set_yscale("log")
axes[1].set_title(r"Distribution of $x_b$", fontsize=16)
axes[1].grid(True, linestyle="--", alpha=0.6)

# Labels 
axes[0].set_xlabel(r"$x_a$", fontsize=14)
axes[1].set_xlabel(r"$x_b$", fontsize=14)
axes[0].set_ylabel("Events / 0.02 ", fontsize=14)

plt.close(fig)   

## Invariant Mass

We start by first plotting the invariant mass from the generation itself as a histogram that is properly normalized. 

Then we plot a (theory) curve using the PDFs directlt (i.e. no MC generation) that pulls a random process from the process map in configurations. This is also normalized. 

Lastly, we overlay the MC generation plot with the theory curve to make sure the two align. 

Note: in MC if we are allowing all active processes, then the two will never directly align. One can check (which I have done a long time ago) that each process is properly following the theory curve by only generating with one active collision process and turning off the random theory choice. 

Plots: saved to Plots/energy_folder/what_process/InvariantMass

In [ ]:
all_M = []
all_weights = []

frame_choice_mass = "CoM" # Choice does not change the invariant mass plots

# Aggregate invariant mass and weights 
for directory in data_dict:
    for lprup, dd in data_dict[directory].items():
        # Skip entries that are not per-lprup dicts
        if not isinstance(dd, dict):
            continue
        # Skip if no invariant-mass list present
        if f"M_{frame_choice_mass}" not in dd:
            continue
        all_M.extend(dd[f"M_{frame_choice_mass}"])
        # Rescale per-slice weights where available (guard against empty weighting)
        wsum = sum(dd.get("weighting", []) )
        if wsum > 0:
            all_weights.extend([w * (total_cross_sections[directory] / wsum) for w in dd.get("weighting", [])])
        else:
            # If no per-event weights, fall back to uniform per-event weighting using xsec totals
            n = len(dd.get(f"M_{frame_choice_mass}", []))
            if n > 0 and total_cross_sections.get(directory,0.0) > 0:
                per = total_cross_sections[directory] / n
                all_weights.extend([per]*n)
            else:
                all_weights.extend([1.0]*len(dd.get(f"M_{frame_choice_mass}", [])))

all_M = np.array(all_M) if all_M else np.array([])
all_weights = np.array(all_weights) if all_weights else np.array([])

if all_M.size == 0:
    raise RuntimeError("No invariant-mass data collected; ensure LHE files were found and parsed.")

num_bins = 70
range_M = (all_M.min(), all_M.max())
y_axis = (all_M.max() - all_M.min()) / num_bins
hist, bin_edges = np.histogram(all_M, bins=num_bins, range=range_M, weights=all_weights if all_weights.size>0 else None)

bin_widths = np.diff(bin_edges)
hist_normalized = hist / bin_widths  # Gives dσ/dM

# Normalize to unity (guard against zero area)
area_hist = np.sum(hist_normalized * bin_widths)
if area_hist == 0:
    hist_unit = hist_normalized
else:
    hist_unit = hist_normalized / area_hist
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Plot histogram
plt.figure(figsize=(9,5))
plt.step(bin_centres, hist_unit, where='mid', linewidth=1.5)
plt.yscale("log")
plt.xlabel("Invariant Mass [GeV]")
plt.ylabel(f"Events/{y_axis:.1f} GeV")
#plt.title("DRToM Curve (Normalized to Unity)")
plt.grid(True)
plt.tight_layout()

# Prepare output folder using unified labels (no gen_type/what_subprocess)
outdir_base = os.path.join("Plots", energy_folder, what_process, "InvariantMass")
os.makedirs(outdir_base, exist_ok=True)
outpath_dr = os.path.join(outdir_base, "InvarMass_DRToM.png")
plt.savefig(outpath_dr, dpi=300)
print("InvarMass_DRToM saved ->", outpath_dr)
# plt.show()

# ====================================
# THEORY CURVES 
# ====================================
theory_curves = {}

s = cfg.s
PDF = fns.PDF
M_min, M_max = all_M.min(), all_M.max()
number_of_points = 200
M_vals = np.linspace(M_min, M_max, number_of_points)

sum_all_IDs = True
# Temporary: select a subprocess at random for theory curve generation
subprocess = np.random.choice(list(cfg.process_map.keys()))
print(f"Selected subprocess for theory curve: {subprocess}")
dSigma = []
combinations = fns.subprocess_combinations(subprocess)

for M in M_vals:
    tau = M**2 / s
    Ymax = min(np.log(1/np.sqrt(tau)), cfg.yMax)

    sigma_list = []
    for ID1, ID2, func in combinations:
        sigma_M = fns.Integrate(
                lambda *args: fns.convolution(*args, M_vals[0], M_vals[-1], func),
                (M, ID1, ID2, s, PDF),
                fns.MC, -Ymax, Ymax, cfg.yMax,
                )
        sigma_list.append(sigma_M)

    sigma_array = np.array(sigma_list)

    if sum_all_IDs:
        sigma_total = np.sum(sigma_array)
    else:
        total_sigma = np.sum(sigma_array)
        if total_sigma == 0:
            dSigma.append(0.0)
            continue
        probs = sigma_array / total_sigma
        chosen_idx = np.random.choice(len(combinations), p=probs)
        sigma_total = sigma_array[chosen_idx]
    sigma_total *= 0.389379e9 * 1e3  # Convert to fb/GeV
    dSigma.append(sigma_total)

theory_curves[subprocess] = np.array(dSigma)


# If theory curves were computed, interpolate and plot; save overlay
if theory_curves:
    # Interpolate theory to same bins
    # Use the first available theory curve for interpolation reference
    first_curve = next(iter(theory_curves.values()))
    theory_interp = np.interp(bin_centres, M_vals, first_curve)
    area_theory = np.sum(theory_interp * bin_widths)
    theory_unit = theory_interp / area_theory
    theory_unit_interp = np.interp(M_vals, bin_centres, theory_unit)

    # Plot theory curves
    plt.figure(figsize=(9,5))
    for name, dSigma in theory_curves.items():
        plt.plot(M_vals, theory_unit_interp, label=name)

    plt.xlabel("Invariant mass [GeV]")
    plt.ylabel("dσ/dM [fb/GeV]")
    plt.title("Theory Curve")
    plt.yscale("log")
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.tight_layout()
    # plt.show()

    # ====================================
    # OVERLAY
    # ====================================
    plt.figure(figsize=(9,5))
    plt.step(bin_centres, hist_unit, where='mid', linewidth=1.5, label="DRToM")
    for name, dSigma in theory_curves.items():
        plt.plot(M_vals, theory_unit_interp, label=name)

    plt.xlabel("Invariant Mass [GeV]")  
    plt.ylabel("dσ/dM [fb/GeV]")
    plt.yscale("log")
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()

    outpath_overlay = os.path.join(outdir_base, "InvarMass_Overlay.png")
    plt.savefig(outpath_overlay, dpi=300)
    print("InvarMass saved ->", outpath_overlay)
    # plt.show()
else:
    print("No theory curves available; overlay not produced.")

plt.close('all')  

InvarMass_DRToM saved -> Plots/TeV13p0/2to2/InvariantMass/InvarMass_DRToM.png
Selected subprocess for theory curve: gg_qqx
InvarMass saved -> Plots/TeV13p0/2to2/InvariantMass/InvarMass_Overlay.png


## Momentum and Angles

Here we plot and save the momentum and angle plots from the lab and CoM. 

Note: the 'modes' here is to filter what part of the process one wants to look at. Right now the loop runs over all modes to make life easier, then one can choose which plots to use. I have not updated this to include $2\to 3$ or $2\to 5$ yet. 

Plots: saved to Plots/energy_folder/what_process/MomentumAndAngles/ then fursther subdirectories depending on the case. 

In [ ]:
mom_angle_folder = os.path.join("Plots", energy_folder, what_process, "MomentumAndAngles")
for folder in [mom_angle_folder]:
    os.makedirs(folder, exist_ok=True)
    
all_delta_eta, all_delta_theta, all_delta_phi = [], [], [] # Collectors for Δη/Δθ/Δφ

if what_process in ["2to2", "2to2_QCD"]:
    modes = ["All", "Leading", "Subleading"]
elif what_process == "2to4":
    modes = ["All", "Leading", "Subleading", "Tertiary", "Last"]
else:
    modes = ["All"]

# Process both frames and save outputs into a <frame> folder
frames = ["lab", "CoM"]
for frame_choice in frames:
    print(f"--- Processing frame: {frame_choice} ---")
    final_frame_folder = os.path.join(mom_angle_folder, frame_choice)
    individual_folder = os.path.join(final_frame_folder, "Individual")
    overlay_folder    = os.path.join(final_frame_folder, "Overlay")
    differences_folder = os.path.join(final_frame_folder, "Differences")

    for folder in [individual_folder, overlay_folder, differences_folder]:
        os.makedirs(folder, exist_ok=True)

    data_by_mode = {}

    # Aggregate four-mom for this frame from directory-level pre-aggregated lists
    aggregated_four_mom = []
    dir_key = f"all_four_mom_{frame_choice}"
    for directory, lprup_dict in data_dict.items():
        dir_list = data_dict[directory].get(dir_key) if isinstance(data_dict[directory], dict) else None
        if isinstance(dir_list, list) and len(dir_list) > 0:
            aggregated_four_mom.extend(dir_list)
            continue
        # Fallback to per-lprup lists
        for lprup, dd in lprup_dict.items():
            if not isinstance(dd, dict):
                continue
            aggregated_four_mom.extend(dd.get(f"four_mom_{frame_choice}", []))

    print(f"✅ Aggregated {len(aggregated_four_mom)} events for frame={frame_choice}.")

    # === Loop over modes for this frame ===
    for mode in modes:
        print(f"plotting to -> {os.path.join(individual_folder, mode)}")
        mode_folder = os.path.join(individual_folder, mode)
        os.makedirs(mode_folder, exist_ok=True)

        # Compute kinematics for the aggregated events
        (three_mom_all, energy_list, momentum_list, pt_list,
         eta_per_event, phi_per_event, px_list, py_list, pz_list,
         theta_per_event, delta_eta_list, delta_theta_list, delta_phi_list) = dr.diff_momentum(aggregated_four_mom, mode=mode)

        # Flatten per-particle values
        energy_flat   = [v for ev in energy_list for v in ev]
        momentum_flat = [v for ev in momentum_list for v in ev]
        pt_flat       = [v for ev in pt_list for v in ev]
        px_flat       = [v for ev in px_list for v in ev]
        py_flat       = [v for ev in py_list for v in ev]
        pz_flat       = [v for ev in pz_list for v in ev]
        eta_flat      = [v for ev in eta_per_event for v in ev]
        phi_flat      = [v for ev in phi_per_event for v in ev]
        theta_flat    = [v for ev in theta_per_event for v in ev]

        # Save Δη/Δθ/Δφ only from 'All'
        if mode == "All":
            all_delta_eta   = delta_eta_list
            all_delta_theta = delta_theta_list
            all_delta_phi   = delta_phi_list

        # Save overlay data per-mode
        data_by_mode[mode] = {
            "energy":   energy_flat,
            "momentum": momentum_flat,
            "pt":       pt_flat,
            "px":       px_flat,
            "py":       py_flat,
            "pz":       pz_flat,
            "eta":      eta_flat,
            "phi":      phi_flat,
            "theta":    theta_flat
        }

    # Overlay plots for this frame
    try:
        dr.plot_kinematics_overlay_full(
            data_by_mode,
            output_file_prefix=os.path.join(overlay_folder, "")
        )
    except Exception as e:
        print(f"Overlay plotting failed for frame={frame_choice}: {e}")

    # Leading–Subleading differences for this frame
    try:
        print(f"plotting to -> {differences_folder}")
        dr.plot_jet_differences(
            all_delta_eta, all_delta_theta, all_delta_phi,
            output_file_prefix=os.path.join(differences_folder, "")
        )
    except Exception as e:
        print(f"Difference plotting failed for frame={frame_choice}: {e}")

plt.close('all') 

--- Processing frame: lab ---
✅ Aggregated 462 events for frame=lab.
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/lab/Individual/All
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/lab/Individual/Leading
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/lab/Individual/Subleading
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/lab/Differences
--- Processing frame: CoM ---
✅ Aggregated 462 events for frame=CoM.
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/CoM/Individual/All
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/CoM/Individual/Leading
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/CoM/Individual/Subleading
plotting to -> Plots/TeV13p0/2to2/MomentumAndAngles/CoM/Differences


## ID Count

In the MC generation, I have for each range a summary output that can be found in the Summary/ directory. In those text files is the information for the incoming and outgoing events that take place in a collision. This code takes those summary files and outputs the plots for incoming particle IDs and outgoing particle IDs.

Plots: saved to /Plots/energy_folder/what_process/Kinematics/

In [ ]:
# Locate summary files under the project Summary tree mirroring the LHE slicing
summary_root = os.path.join(os.path.dirname(base_path), "Summary", energy_folder)
summary_files = []
print(f"Searching for summaries under: {summary_root}")
for slice in np.arange(Start, End, Step):
    s = round(slice, 10)
    e = round(slice + Step, 10)
    label = _slice_label(s, e)
    # search both possible dimensionality folders (2D and 3D) recursively
    for dim in [dimensionality_3D, dimensionality_2D]:
        pattern = os.path.join(summary_root, dim, "**", what_process, f"summary_{label}*.txt")
        matches = sorted(glob.glob(pattern, recursive=True))
        if matches:
            summary_files.extend(matches)

# Deduplicate and sort
summary_files = sorted(set(summary_files))
print(f"Found {len(summary_files)} summary files under {summary_root} for process {what_process}.")

summary_output_path = os.path.join("Plots", energy_folder, what_process, "Kinematics", "summary_output.txt")
os.makedirs(os.path.dirname(summary_output_path), exist_ok=True)

# Combine into the summary_output used by downstream plotting
if summary_files:
    os.makedirs(os.path.dirname(summary_output_path), exist_ok=True)
    with open(summary_output_path, "w") as out:
        for fname in summary_files:
            with open(fname, "r") as inp:
                out.write(inp.read())
                out.write("\n")
    print(f"Combined summary written -> {summary_output_path}")
else:
    print("No summary files found; using existing summary_output.txt if present.")

# Parse and plot using the combined summary
in_counts = dr.parse_summary_output(summary_output_path)
in_outpath = os.path.join("Plots", energy_folder, what_process, "Kinematics", "IncomingIDs.png")
os.makedirs(os.path.dirname(in_outpath), exist_ok=True)
dr.plot_initial_ids_counts(in_counts, what_process, outpath=in_outpath)
print(f"Incoming IDs plot saved -> {in_outpath}")

out_counts = dr.parse_summary_output_outgoing(summary_output_path)
out_outpath = os.path.join("Plots", energy_folder, what_process, "Kinematics", "OutgoingIDs.png")
os.makedirs(os.path.dirname(out_outpath), exist_ok=True)
dr.plot_outgoing_ids_counts(out_counts, what_process, outpath=out_outpath)
print(f"Outgoing IDs plot saved -> {out_outpath}")

plt.close('all')

Searching for summaries under: /hepusers2/fuscomus/DRToM/Summary/TeV13p0
Found 40 summary files under /hepusers2/fuscomus/DRToM/Summary/TeV13p0 for process 2to2.
Combined summary written -> Plots/TeV13p0/2to2/Kinematics/summary_output.txt
Incoming IDs plot saved -> Plots/TeV13p0/2to2/Kinematics/IncomingIDs.png
Outgoing IDs plot saved -> Plots/TeV13p0/2to2/Kinematics/OutgoingIDs.png


## Event Shape 

Lastly, we now run through our event shape variables in the lab and CoM frame. The plotting output was arbitrary. 

Plots: saved to Plots/energy_folder/what_process/EventShapeVars

In [ ]:
import matplotlib.gridspec as gridspec

shape_vars = [
    "aplanarity", "B_values",
    "sphericity", "sphericity_transverse",
    "Y_values", "C_values", "D_values",
    "Thrust_T_values", "Thrust_m_values", "tau_values"
]

x_labels = {
    "sphericity": "Sphericity (S)",
    "aplanarity": "Aplanarity (A)",
    "sphericity_transverse": "Transverse Sphericity ($S_T$)",
    "Y_values": "Y Parameter",
    "C_values": "C Parameter",
    "D_values": "D Parameter",
    "Thrust_T_values": "Transverse Thrust ($T_T$)",
    "Thrust_m_values": "Major Thrust ($T_m$)",
    "tau_values": "$\\tau \ \ (= 1 - T$)",
    "B_values": "Biplanarity (B)",
}

plot_config = {
    "sphericity": {"range": (0, 1), "bins": 50},
    "aplanarity": {"range": (0, 0.5), "bins": 50},
    "sphericity_transverse": {"range": (0, 1), "bins": 50},
    "Y_values": {"range": (0, 1), "bins": 50},
    "C_values": {"range": (0, 1), "bins": 50},
    "D_values": {"range": (0, 1), "bins": 50},
    "Thrust_T_values": {"range": (2/np.pi, 1), "bins": 50},
    "Thrust_m_values": {"range": (0, 2/np.pi), "bins": 50},
    "tau_values": {"range": (0, 1 - 2/np.pi), "bins": 50},
    "B_values": {"range": (0, 1), "bins": 50},
}

row_cols = [2, 2, 3, 3]
row_names = ["AB", "Sphericity", "Letters", "Thrust"]

def flatten_vals(vals):
    if vals is None:
        return np.array([])
    out = []
    for v in vals:
        if isinstance(v, np.ndarray):
            if v.ndim == 0:
                out.append(v.item())
            else:
                out.extend(v.tolist())
        elif isinstance(v, (list, tuple)):
            out.extend(v)
        else:
            out.append(v)
    return np.array(out)

# Store both frames for overlay later
all_frames_data = {}

# === Step 1: Loop Over Frames === 
for frame_choice in ["lab", "CoM"]:
    print(f"\n--- Event shape: frame={frame_choice} ---")
    suffix = "_lab" if frame_choice == "lab" else "_CoM"

    aggregated = {var: [] for var in shape_vars}
    for directory, lprup_dict in data_dict.items():
        for _, dd in lprup_dict.items():
            if not isinstance(dd, dict):
                continue
            for var in shape_vars:
                vals = dd.get(var + suffix, [])
                if isinstance(vals, (list, tuple, np.ndarray)):
                    aggregated[var].extend(vals)
                else:
                    aggregated[var].append(vals)

    # flatten once here (important for reuse)
    aggregated = {k: flatten_vals(v) for k, v in aggregated.items()}
    all_frames_data[frame_choice] = aggregated

    # === Full Grid ===
    fig = plt.figure(figsize=(20, 18))
    plt.rcParams.update({'font.size': 11})

    max_cols = max(row_cols)
    gs = gridspec.GridSpec(len(row_cols), max_cols, figure=fig, hspace=0.35, wspace=0.3)

    var_idx = 0
    for row, n_cols in enumerate(row_cols):
        for col in range(n_cols):
            ax = fig.add_subplot(gs[row, col])
            var = shape_vars[var_idx]
            data = aggregated[var]

            if data.size == 0:
                ax.text(0.5, 0.5, "No data", ha="center", va="center")
                ax.axis("off")
            else:
                cfg = plot_config.get(var, {})
                hist_range = cfg.get("range", None)
                bins = cfg.get("bins", 60)

                # === Violation Check ===
                if hist_range is not None:
                    outside = data[(data < hist_range[0]) | (data > hist_range[1])]
                    if outside.size > 0:
                        print(f"[WARNING] {var} ({frame_choice}): {outside.size} values outside {hist_range}")

                counts, bins_arr, _ = ax.hist(
                    data,
                    bins=bins,
                    range=hist_range,
                    density=False,   # This normalizes to total count, not area
                    color="steelblue",
                    edgecolor="black",
                    alpha=0.75
                )

                if hist_range is not None:
                    ax.set_xlim(hist_range)

                ax.set_xlabel(x_labels.get(var, var))
                if len(bins_arr) > 1:
                    bin_width = bins_arr[1] - bins_arr[0]
                    ax.set_ylabel(f"Events / {bin_width:.3f}")
                else:
                    ax.set_ylabel("Events")
                ax.grid(alpha=0.2)

            var_idx += 1

    plt.tight_layout()

    outdir = os.path.join("Plots", energy_folder, what_process, "EventShapeVars")
    os.makedirs(outdir, exist_ok=True)
    plt.savefig(os.path.join(outdir, f"EventShapeVars_full_{frame_choice}.png"), dpi=300)
    plt.close()


# === Step 2: Overlay Plots (lab vs CoM) ===
print("\n--- Creating overlay plots (lab vs CoM) ---")

overlay_dir = os.path.join("Plots", energy_folder, what_process, "EventShapeVars", "overlay")
os.makedirs(overlay_dir, exist_ok=True)

for var in shape_vars:
    data_lab = all_frames_data["lab"][var]
    data_com = all_frames_data["CoM"][var]

    if data_lab.size == 0 and data_com.size == 0:
        continue

    cfg = plot_config.get(var, {})
    hist_range = cfg.get("range", None)
    bins = cfg.get("bins", 60)

    plt.figure(figsize=(6, 5))

    # lab
    plt.hist(
        data_lab,
        bins=bins,
        range=hist_range,
        density=True,
        histtype="step",
        linewidth=2,
        label="lab"
    )

    # CoM
    plt.hist(
        data_com,
        bins=bins,
        range=hist_range,
        density=True,
        histtype="step",
        linewidth=2,
        linestyle="--",
        label="CoM"
    )

    if hist_range is not None:
        plt.xlim(hist_range)

    plt.xlabel(x_labels.get(var, var))
    if len(bins_arr) > 1:
        bin_width = bins_arr[1] - bins_arr[0]
        ax.set_ylabel(f"Events / {bin_width:.3f}")
    else:
        ax.set_ylabel("Events")
    plt.legend()
    plt.grid(alpha=0.2)

    outpath = os.path.join(overlay_dir, f"{var}_overlay.png")
    plt.savefig(outpath, dpi=300)
    plt.close()

    print("Saved overlay ->", outpath)


--- Event shape: frame=lab ---
[WARNING] B_values (lab): 462 values outside (0, 1)
[WARNING] sphericity_transverse (lab): 458 values outside (0, 1)
[WARNING] Thrust_T_values (lab): 113 values outside (0.6366197723675814, 1)
[WARNING] tau_values (lab): 113 values outside (0, 0.3633802276324186)


/tmp/ipykernel_3112806/1839628660.py:130: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



--- Event shape: frame=CoM ---
[WARNING] B_values (CoM): 462 values outside (0, 1)
[WARNING] sphericity_transverse (CoM): 458 values outside (0, 1)
[WARNING] Thrust_T_values (CoM): 113 values outside (0.6366197723675814, 1)
[WARNING] tau_values (CoM): 113 values outside (0, 0.3633802276324186)

--- Creating overlay plots (lab vs CoM) ---
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/aplanarity_overlay.png


/hepusers2/fuscomus/miniconda3/envs/DRToM/lib/python3.10/site-packages/numpy/lib/_histograms_impl.py:901: RuntimeWarning: invalid value encountered in divide
  return n/db/n.sum(), bin_edges


Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/B_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/sphericity_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/sphericity_transverse_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/Y_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/C_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/D_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/Thrust_T_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/Thrust_m_values_overlay.png
Saved overlay -> Plots/TeV13p0/2to2/EventShapeVars/overlay/tau_values_overlay.png
